In [5]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 5  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line): 

John Mandalas

Mohammadreza Babaeizadeh

Chloe Pang

Pranav Sudev

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

## Variables

Global variables that can be configured.


In [ ]:
# Client Configurations
client_a_budget = 1000000
client_b_budget = 750000

min_port_size = 10
max_port_size = 20

min_weight = 0.15
max_weight = 0.40
max_stock_weight = 0.15

# Stock eligibility
csv_file_attach = "Tickers_Example.csv"
min_avg_d_vol = 5000

start_date = "2024-10-01"
end_date = "2026-11-28"

min_trade_d_per_m = 18

# Purchase and transaction fees in USD

#purchase_date = str(stock_prices.index[-1].date())   # testing only
purchase_date = "2026-11-25"

fee_cap_per_stock = 2.15
fee_per_share = 0.001  

# Bank of Canada 3-month Treasury Bill Rate (as of 2026-11-25)
risk_free_rate = 0.0269

In [7]:
# Gets all the list of tickers from the csv file and returns them as a pandas dataframe
# Parameters: file_name (str) - the name of the csv file containing the tickers
# Returns: pandas dataframe containing the tickers
def get_listof_tickers(file_name):
    listof_tickers = pd.read_csv(file_name, header=None)
    listof_tickers = listof_tickers.drop_duplicates().reset_index(drop=True)
    return listof_tickers


# Normalizes the list of tickers to ensure they fit with the assignment criteria
# Parameters: tickers (pandas dataframe) - the dataframe containing the tickers to normalize
# Returns: pandas dataframe containing the normalized tickers
def normalize_listof_tickers(tickers):
    stock_data = []

    valid_exchanges = ["NMS", "NYQ", "ASE", "TOR", "VAN", "NEO"]# TBDTBDTBD
    
    # Get data needed for each ticker in the dataframe.
    for ticker in tickers[0]:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            history = stock.history(start="2025-10-01", end="2026-10-01", auto_adjust=False) # Last trading date checked: 2026-09-30

            # Skip stocks without historical data
            if history.empty:
                print("Not Valid Ticker (no data):", ticker)
                continue

            # Check if the stock meets the basic requirements
            valid_stock = (info.get("quoteType") == "EQUITY" 
                           and info.get("currency") in ["CAD", "USD"] 
                           and info.get("exchange") in valid_exchanges)
           
            # Counting trading days in each calendar month
            month_counts = {}

            # Go through all dates and count trading days for each month
            for date in history.index:
                month = date.month

                if month in month_counts:
                    month_counts[month] += 1
                else:
                    month_counts[month] = 1

            lowest_trading_day = min(month_counts.values())

            # Calculate average daily trading volume
            total_volume = 0
            total_days = 0
            
            for date, row in history.iterrows():
                month = date.month
                
                # Only include months with at least 18 trading days
                if month_counts[month] >= min_trade_d_per_m: 
                    total_volume += row["Volume"]
                    total_days += 1
            
            if total_days > 0:
                average_volume = total_volume / total_days
            else:
                average_volume = 0

            # Creating a dataframe containing information needed in further functions
            if valid_stock and average_volume >= min_avg_d_vol:
                stock_data.append({
                    "Ticker": ticker,
                    "Stock's Name": info.get("longName"),
                    "Exchange": info.get("exchange"),
                    "Currency": info.get("currency"),
                    "Ticker Type": info.get("quoteType"),
                    "Sector": info.get("sector"),
                    "Monthly Trading Days": lowest_trading_day,
                    "Average Volume": average_volume
            })

        # Mark the stock as invalid if an error occurs
        except Exception as e:
            print("Not Valid Ticker:", ticker, "-", e)

    return pd.DataFrame(stock_data)


# Use the new list that only contains the valid data (Not necessary to produce invalid data)
# Returns: a dataframe cointaining the {TICKER}_CLOSE and {TICKER}_ADJ_CLOSE for each ticker in the list of tickers, for the given start and end dates
def create_dataframe(listof_tickers, start_date, end_date):
    stock_prices = pd.DataFrame()
    for ticker in listof_tickers["Ticker"]: # Change [0] to ["Ticker"] just to get the valid stocks price
        stock = yf.Ticker(ticker)
        history = stock.history(start=start_date, end=end_date,auto_adjust=False)

        if not history.empty:
            history.index = history.index.tz_localize(None)
            prices = history[["Close", "Adj Close"]].copy() #Add other categories to here
            prices.columns = [ticker + "_Close", ticker + "_Adj_Close"]
            stock_prices = pd.concat([stock_prices, prices], axis=1,sort=True)

    return stock_prices


# Gets the USD to CAD exchange rate between two dates using yfinance
# Parameters: start_date (str) - the start date to get the exchange rate for in the format 'YYYY-MM-DD'
# end_date (str) - the end date to get the exchange rate for in the format 'YYYY-MM-DD'
# Returns: float - the USD to CAD exchange rate for the given date
def get_USD_to_CAD_exchange_rate(start_date, end_date):
    # Get the USD to CAD exchange rate for the given date using yfinance
    exchange_rate = yf.download('CAD=X', start=start_date, end=end_date)
    return exchange_rate['Close'].squeeze("columns")

# Buys a ticker and adds it to the portfolio, including fees
# Parameters: ticker_name (str) - the ticker to buy, dollars (float) - the amount of CAD to spend including fees,
# portfolio (pandas dataframe) - the portfolio to add the ticker to, date (str) - the purchase date 'YYYY-MM-DD'
# Returns: num_shares (float) - the number of shares bought
def buy_ticker(ticker_name, dollars, portfolio, date):
    rate = exchange_rates.loc[date]
    price = stock_prices.loc[date, ticker_name + "_Close"]

    # Convert USD prices to CAD
    if normalized_tickers.set_index("Ticker").loc[ticker_name, "Currency"] == "USD":
        price = price * rate

    # Per-share fee case: dollars = shares * price + shares * 0.001 * rate
    num_shares = dollars / (price + rate * 0.001)

    # If the per-share fee would be more than the flat fee, the flat fee applies
    if calculate_transaction_fees(num_shares, rate) < num_shares * rate * 0.001:
        num_shares = (dollars - rate * 2.15) / price

    portfolio.loc[ticker_name, "Shares"] = num_shares
    return num_shares


# Calculates volatility of the whole portfolio over a given time period using the standard deviation of the returns
# Parameters: daily_returns (pandas series), the series that contains the daily returns of the portfolio over a given time period
# Returns: the volatility of the portfolio 
def calculate_volatility(daily_returns):
    return daily_returns.std()

# Calculate transaction fees for buying a ticker
# Parameters: shares (float), the number of shares to buy
# exchange_rate (float), CAD per US$1, the exchange rate for the given date
# Returns: the transaction fees for the stock purchase (in CAD)
def calculate_transaction_fees(shares, exchange_rate):
    fees = min(exchange_rate * 2.15, shares * exchange_rate * 0.001)
    return fees

# Calculates the value of a portfolio on a given date in CAD
# Parameters: portfolio (pandas dataframe) - the portfolio, date (str) - the date 'YYYY-MM-DD'
# Returns: the total value of the portfolio in CAD
def portfolio_value(portfolio, date):
    rate = exchange_rates.loc[date]
    total = 0
    for ticker in portfolio.index:
        price = stock_prices.loc[date, ticker + "_Close"]
        if normalized_tickers.set_index("Ticker").loc[ticker, "Currency"] == "USD":
            price = price * rate
        total += portfolio.loc[ticker, "Shares"] * price
    return total


# Calculates the portfolio's return over the comparison period
# Parameters: portfolio (pandas dataframe) - the portfolio, start_date (str) - start of the period, end_date (str) - end of the period
# Returns: the portfolio's return as a decimal (0.05 = 5%)
def calculate_portfolio_return(portfolio, start_date, end_date):
    return portfolio_value(portfolio, end_date) / portfolio_value(portfolio, start_date) - 1


# Convert USD stock prices to CAD using the exchange rate for each given date
# Parameters: stock_prices (pandas dataframe), the dataframe that contains the stock prices for each ticker in USD
# exchange_rates (pandas series), the series that contains the exchange rates for each date
# Returns: the pandas dataframe that contains the stock prices for each ticker in CAD
def convert_usd_to_cad(stock_prices, exchange_rates, normalized_tickers):
    converted_prices = stock_prices.copy()
    for i, stock in normalized_tickers.iterrows():
        ticker = stock["Ticker"]
        if stock["Currency"] == "USD": 
            for column in [ticker + "_Close", ticker + "_Adj_Close"]:
                if column in converted_prices.columns:
                    converted_prices[column] = converted_prices[column] * exchange_rates
    return converted_prices


# Calculate the historical portfolio value over a given time period (CAD)
# Parameters: stock_prices (pandas dataframe), the dataframe that contains the stock prices for each ticker (CAD)
# portfolio (pandas dataframe), the dataframe that contains the tickers and number of shares in the portfolio
# Returns: the pandas series that contains the historical portfolio value over the given time period
def calculate_portfolio_value(stock_prices, portfolio):
    portfolio_values = pd.Series(0.0, index=stock_prices.index)
    for ticker in portfolio.index:
        shares = portfolio.loc[ticker, "Shares"]
        if ticker + "_Close" in stock_prices.columns:
            portfolio_values += stock_prices[ticker + "_Close"] * shares
    return portfolio_values



# Calculate the daily returns of the portfolio over a given time period
# Parameters: portfolio_values (pandas series), the series that contains the historical portfolio value over the given time period
# Returns: the pandas series that contains the daily returns of the portfolio over the given time period
def calculate_daily_returns(portfolio_values):
    daily_returns = portfolio_values.pct_change().dropna()
    return daily_returns


    
            


"""
TO DO:

Beta calculation function
"""

# Calling ticker list
listof_tickers = get_listof_tickers(csv_file_attach)
display(listof_tickers)

# Calling valid ticker 
normalized_tickers = normalize_listof_tickers(listof_tickers)
display(normalized_tickers)

# Calling create_dataframe for Adj and Close price
stock_prices = create_dataframe(normalized_tickers, start_date, end_date)
stock_prices = stock_prices.ffill()   # holiday: use the last close
display(stock_prices)

# Calling the exchange_rates USD CAD
exchange_rates = get_USD_to_CAD_exchange_rate(start_date, end_date)
exchange_rates = exchange_rates.reindex(stock_prices.index).ffill()
display(exchange_rates)

,0
0,AAPL
1,ABBV
2,ABT
3,ACN
4,AGN
5,AIG
6,AMZN
7,AXP
8,BA
9,BAC


$AGN: possibly delisted; no timezone found


Not Valid Ticker (no data): AGN


$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)


Not Valid Ticker (no data): BK


$CELG: possibly delisted; no timezone found


... (5 outputs left out)

,Ticker,Stock's Name,Exchange,Currency,Ticker Type,Sector,Monthly Trading Days,Average Volume
0,AAPL,Apple Inc.,NMS,USD,EQUITY,Technology,19,4.828203e+07
1,ABBV,AbbVie Inc.,NYQ,USD,EQUITY,Healthcare,19,6.281745e+06
2,ABT,Abbott Laboratories,NYQ,USD,EQUITY,Healthcare,19,1.008423e+07
3,ACN,Accenture plc,NYQ,USD,EQUITY,Technology,19,6.116120e+06
4,AIG,"American International Group, Inc.",NYQ,USD,EQUITY,Financial Services,19,4.324195e+06
5,AMZN,"Amazon.com, Inc.",NMS,USD,EQUITY,Consumer Cyclical,19,4.652561e+07
6,AXP,American Express Company,NYQ,USD,EQUITY,Financial Services,19,3.054224e+06
7,BA,The Boeing Company,NYQ,USD,EQUITY,Industrials,19,7.140954e+06
8,BAC,Bank of America Corporation,NYQ,USD,EQUITY,Financial Services,19,3.673368e+07
9,BB.TO,BlackBerry Limited,TOR,CAD,EQUITY,Technology,19,3.406802e+06


,AAPL_Close,AAPL_Adj_Close,ABBV_Close,ABBV_Adj_Close,ABT_Close,ABT_Adj_Close,ACN_Close,ACN_Adj_Close,AIG_Close,AIG_Adj_Close,...,TXN_Close,TXN_Adj_Close,UNH_Close,UNH_Adj_Close,UNP_Close,UNP_Adj_Close,UPS_Close,UPS_Adj_Close,USB_Close,USB_Adj_Close
Date,,,,,,,,,,,,,,,,,,,,,
2024-10-01,226.210007,224.163101,196.960007,183.311874,113.559998,108.235291,351.980011,335.945465,72.720001,69.465912,...,201.600006,189.847687,583.299988,556.110413,245.550003,234.895584,133.270004,115.883911,44.459999,40.971088
2024-10-02,226.779999,224.727921,196.820007,183.181580,113.639999,108.311546,356.290009,340.059113,72.449997,69.207993,...,203.429993,191.570999,592.309998,564.700562,243.350006,232.791061,133.009995,115.657791,44.250000,40.777569
2024-10-03,225.669998,223.627960,195.449997,181.906525,112.230003,106.967667,361.239990,344.783600,72.339996,69.102913,...,201.669998,189.913605,592.750000,565.119934,240.630005,230.189072,130.720001,113.666550,43.980000,40.528755
2024-10-04,226.800003,224.747772,194.289993,180.826889,112.639999,107.358437,362.239990,345.738007,76.050003,72.646896,...,202.710007,190.892990,591.200012,563.642273,238.860001,228.495865,131.250000,114.127411,44.700001,41.192249
2024-10-07,221.690002,219.684006,193.610001,180.194016,113.360001,108.044678,356.799988,340.545868,73.639999,70.344742,...,201.270004,189.536926,583.559998,556.358398,236.869995,226.592209,131.199997,114.083931,44.470001,40.980305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-11-20,342.559998,342.559998,286.200012,286.200012,103.000000,103.000000,221.169998,221.169998,79.680000,79.680000,...,263.750000,263.750000,339.850006,339.850006,287.290009,287.290009,106.220001,106.220001,56.049999,56.049999
2026-11-23,349.079987,349.079987,288.450012,288.450012,103.320000,103.320000,222.289993,222.289993,80.559998,80.559998,...,291.190002,291.190002,336.079987,336.079987,288.890015,288.890015,108.510002,108.510002,57.410000,57.410000
2026-11-24,354.070007,354.070007,300.279999,300.279999,105.389999,105.389999,232.320007,232.320007,81.239998,81.239998,...,258.440002,258.440002,341.880005,341.880005,290.829987,290.829987,110.830002,110.830002,58.070000,58.070000


[*********************100%***********************]  1 of 1 completed


Date
2024-10-01    1.35272
2024-10-02    1.34946
2024-10-03    1.35049
2024-10-04    1.35482
2024-10-07    1.35792
               ...   
2026-11-20    1.43435
2026-11-23    1.43388
2026-11-24    1.43653
2026-11-25    1.42729
2026-11-26    1.42420
Name: CAD=X, Length: 553, dtype: float64

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [ ]:
# Convert all USD stock prices to CAD
cad_prices = convert_usd_to_cad(stock_prices, exchange_rates, normalized_tickers)

# Get S&P 500 prices and convert them to CAD
market = create_dataframe(pd.DataFrame({"Ticker": ["^GSPC"]}), start_date, end_date)["^GSPC_Close"]
market = market.reindex(stock_prices.index)
market = market.dropna()
market = market * exchange_rates

# Weekly market returns
market_weekly = market.resample("W-FRI").last() #Makes the market data weekly instead of daily because of weekly competition
market_returns = market_weekly.pct_change().dropna()
expected_market_returns= market_returns.mean() * 52 #Annualized expected return of the market

# Initialize betas as a series to store data into (switched from dictionary last commit)
betas = pd.Series()

#Beta of each stock (Put this into the above code section for part b calculations)
for ticker in normalized_tickers["Ticker"]:

    stock_weekly = cad_prices[ticker + "_Close"].resample("W-FRI").last()
    stock_returns = stock_weekly.pct_change().dropna()
    covariance = stock_returns.cov(market_returns)
    variance = market_returns.var()
    beta = covariance / variance
    betas[ticker] = beta

betas = betas.dropna()
betas = betas.sort_values(ascending=False)
display(betas)

# Use CAPM to calculate expected returns for each stock, and then sort from highest expected return to lowest
expected_returns = risk_free_rate + betas * (expected_market_returns - risk_free_rate)
expected_returns = expected_returns.sort_values(ascending=False)
display(expected_returns)

# Top 6 get 15%, last 4 split the remaining 10%
weight_plan = [0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.025, 0.025, 0.025, 0.025]

# Initialize series and variables for stock weight calculation
sectors = normalized_tickers.set_index("Ticker")["Sector"]
a_weights = pd.Series()
sector_weights = pd.Series()
carry = 0

for ticker in expected_returns.index:
    sector = sectors[ticker]
    current = sector_weights.get(sector, 0)
    room = round(max_weight - current, 6)

    if room <= 0:
        continue

    want = weight_plan[len(a_weights)] + carry
    given = min(want, room, max_stock_weight)

    a_weights[ticker] = given
    sector_weights[sector] = current + given
    carry = want - given

    if len(a_weights) == 10:
        break


# Buy each stock with its share of the budget (fees included)
a_tickers = a_weights.index.tolist()

portfolio_a = pd.DataFrame(index=a_tickers, columns=["Shares"])


for ticker in a_tickers:
    buy_ticker(ticker, a_weights[ticker] * client_a_budget, portfolio_a, purchase_date)

# Show portfolio a
portfolio_a["Target Weight"] = pd.Series(a_weights)

portfolio_a["Beta"] = betas[a_tickers]
portfolio_a["Expected Return"] = expected_returns[a_tickers]
portfolio_a["Sector"] = sectors[a_tickers]

display(portfolio_a)
display(pd.Series(sector_weights, name="Sector Weight"))

# Initialize beta variable
beta_max = 0

# Calculate the weighted average of beta (combined beta)
for ticker in a_tickers:
    beta_max += a_weights[ticker] * betas[ticker]

portfolio_expected_return = risk_free_rate + beta_max * (expected_market_returns - risk_free_rate)
print("Expected return of the portfolio is: " + str(portfolio_expected_return))

# Calculate maximum returns
returns_max = calculate_daily_returns(calculate_portfolio_value(cad_prices, portfolio_a))

# Display basic data calculated above. (comment out during actual run)
print("Value on purchase date (CAD): " + str(portfolio_value(portfolio_a, purchase_date)))
print("Portfolio beta: " + str(beta_max))
print("Historical daily volatility: " + str(calculate_volatility(returns_max)))
print("Historical return: " + str(calculate_portfolio_return(portfolio_a, returns_max.index[0], returns_max.index[-1])))

SHOP.TO    2.110361
QCOM       1.618383
BB.TO      1.538128
C          1.495067
BA         1.439559
CAT        1.336810
TXN        1.316856
AMZN       1.250920
BAC        1.165748
BLK        1.143306
AXP        1.110396
USB        1.034076
AAPL       0.973436
PYPL       0.937766
UPS        0.758596
ACN        0.599669
UNH        0.595534
UNP        0.576669
LLY        0.537721
TD.TO      0.503925
RY.TO      0.500456
BIIB       0.469630
MRK        0.464375
PFE        0.459405
AIG        0.438970
LMT        0.379402
BMY        0.377141
PEP        0.210344
ABT        0.200213
CL         0.179703
PM         0.166664
KO         0.162537
ABBV       0.156125
MO         0.048407
T.TO      -0.493583
dtype: float64

SHOP.TO    0.267848
QCOM       0.211677
BB.TO      0.202514
C          0.197598
BA         0.191260
CAT        0.179529
TXN        0.177251
AMZN       0.169723
BAC        0.159998
BLK        0.157436
AXP        0.153678
USB        0.144965
AAPL       0.138041
PYPL       0.133969
UPS        0.113512
ACN        0.095367
UNH        0.094895
UNP        0.092741
LLY        0.088294
TD.TO      0.084435
RY.TO      0.084039
BIIB       0.080520
MRK        0.079920
PFE        0.079352
AIG        0.077019
LMT        0.070218
BMY        0.069960
PEP        0.050916
ABT        0.049759
CL         0.047417
PM         0.045929
KO         0.045457
ABBV       0.044725
MO         0.032427
T.TO      -0.029454
dtype: float64

,Shares,Target Weight,Beta,Expected Return,Sector
SHOP.TO,790.841317,0.150,2.110361,0.267848,Technology
QCOM,623.109492,0.150,1.618383,0.211677,Technology
BB.TO,8284.750112,0.100,1.538128,0.202514,Technology
C,927.32148,0.150,1.495067,0.197598,Financial Services
BA,638.750589,0.150,1.439559,0.191260,Industrials
CAT,193.297796,0.150,1.336810,0.179529,Industrials
AMZN,214.977387,0.075,1.250920,0.169723,Consumer Cyclical
BAC,360.843652,0.025,1.165748,0.159998,Financial Services
BLK,15.339619,0.025,1.143306,0.157436,Financial Services
AXP,56.069833,0.025,1.110396,0.153678,Financial Services


Technology            0.400
Financial Services    0.225
Industrials           0.300
Consumer Cyclical     0.075
Name: Sector Weight, dtype: float64

Expected return of the portfolio is: 0.20194541872038943
Value on purchase date (CAD): 999991.478292251
Portfolio beta: 1.533145100056399
Historical daily volatility: 0.017666923898014466
Historical return: 0.476167266585243


**IMPORTANT READ - REZA**

Reza if you're reading this. I adjusted my strategy to just pick the highest weekly beta stocks and maximize top to bottom of the top 10, and if going over, roll over into the next stock. I did this because, although a different strategy would probably be better, I thought this is the way to get full marks because I did exactly what the question told me to do, which is to maximize expected return. Therefore I had to pick the highest beta stocks in top to bottom order, adjusting for the 15% constraint and the sector cap limit. 

Let me know if this is not a good strategy for the assignment or not. - John

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

'\n\nStrategy for buying stocks:\n\n20 stocks\nChloe\n\n'

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.